# Exploring Data with Pandas

In the real world, data comes in tables — spreadsheets, CSV files, database exports. **Pandas** is Python's most popular library for working with this kind of tabular data.

In this notebook, you'll learn how to:
- Load a dataset from a file
- Inspect and summarize it
- Select, filter, and sort data

We'll work with the [Gapminder](https://www.gapminder.org/) dataset, which tracks **life expectancy**, **population**, and **GDP per capita** for countries around the world from 1952 to 2007. This is the kind of tabular data that scientists work with every day.

*This notebook is inspired by the [Software Carpentry](https://swcarpentry.github.io/python-novice-gapminder/08-data-frames.html) lesson on Pandas DataFrames.*

In [ ]:
import pandas as pd

---
## 1. Loading Data

The most common way to get data into pandas is `pd.read_csv()`. It reads a **CSV** (comma-separated values) file and returns a **DataFrame** — pandas' core data structure for tables.

CSV files are plain text files where each row is a line and columns are separated by commas. You can open them in any text editor or spreadsheet program.

Our dataset is hosted online, so we can pass the URL directly.

In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/gapminder_unfiltered.csv"
df = pd.read_csv(url)

We now have a DataFrame called `df`. Let's take a first look.

> **Important:** Every cell in this notebook depends on `df` existing in memory. If you ever get `NameError: name 'df' is not defined`, it means the cells above (import pandas and load the CSV) haven't been run yet. A safe habit: when you open a notebook, run all cells from the top before jumping around.

### Peeking at the data

`.head()` shows the first 5 rows (or pass a number to see more). `.tail()` shows the last rows.

In [ ]:
df.head()

In [ ]:
df.tail(3)

### How big is the dataset?

`.shape` returns `(rows, columns)`. `.columns` lists the column names.

In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))

### Data types and missing values

`.info()` gives a concise summary: column names, data types, and whether any values are missing. `.dtypes` shows just the types.

In [ ]:
df.info()

In [ ]:
df.dtypes

The columns are:
- **country** — name of the country (`object` = text/string)
- **continent** — which continent (`object`)
- **year** — year of the observation (`int64` = integer)
- **lifeExp** — life expectancy at birth, in years (`float64` = decimal number)
- **pop** — population (`int64`)
- **gdpPercap** — GDP per capita in US dollars, inflation-adjusted (`float64`)

No missing values — this is a clean dataset. Real-world data is rarely this tidy!

### Exercise 1: First look

*If you get a `NameError` saying `df` is not defined, scroll up and run the import and data-loading cells first.*

**1a.** How many rows and columns does the dataset have? *(Use `.shape`)*

In [ ]:
# Your code here


**1b.** How many unique countries are in the dataset?

*Hint: try `df["country"].nunique()`*

In [ ]:
# Your code here


**1c.** What years does the dataset cover?

*Hint: try `df["year"].unique()` or `.min()` and `.max()`*

In [ ]:
# Your code here


---
## 2. Summarizing Data

Once you've loaded a dataset, the next step is usually to compute some **summary statistics** to understand what you're working with.

### `.describe()` — a quick statistical overview

This gives you the count, mean, standard deviation, min, max, and quartiles for every numeric column.

In [ ]:
df.describe()

Look at the **population** column — the mean is about 30 million, but the max is over 1.3 billion. That's a huge range! The standard deviation being larger than the mean tells us the data is very spread out.

### Individual summary statistics

You can also compute statistics one at a time, on the whole DataFrame or on a single column.

In [ ]:
print("Mean life expectancy:", df["lifeExp"].mean())
print("Median life expectancy:", df["lifeExp"].median())
print("Max GDP per capita:", df["gdpPercap"].max())
print("Min GDP per capita:", df["gdpPercap"].min())

### Counting categories

For text columns, `.value_counts()` shows how many times each value appears.

In [ ]:
df["continent"].value_counts()

### Exercise 2: Summary statistics

**2a.** What is the standard deviation of GDP per capita across the whole dataset?

*Hint: use `.std()`*

In [ ]:
# Your code here


**2b.** How many observations (rows) are there for each continent?

*Hint: use `.value_counts()` on the continent column.*

In [ ]:
# Your code here


---
## 3. Selecting Columns and Rows

Most of the time you don't need the entire DataFrame — you want specific columns or rows.

### Selecting columns

Use square brackets with the column name (as a string) to get a single column. The result is a **Series** — a one-dimensional array with an index.

In [ ]:
life_exp = df["lifeExp"]
print(type(life_exp))
life_exp.head()

To select **multiple columns**, pass a list of column names. The result is a DataFrame.

In [ ]:
subset = df[["country", "year", "lifeExp"]]
subset.head()

Notice the double brackets: the outer `[]` is the selection operator, and the inner `[]` creates a list.

### Selecting rows with `.loc` and `.iloc`

Pandas provides two main ways to select rows:

| Method | Selects by | Example |
|---|---|---|
| `.iloc[]` | **position** (integer index) | `.iloc[0]` = first row |
| `.loc[]` | **label** (row/column names) | `.loc[0, "country"]` = row 0, country column |

Think of it this way: **`iloc`** = **i**nteger position, **`loc`** = **l**abel.

In [ ]:
# First row by position
df.iloc[0]

In [ ]:
# Rows 5 through 9 by position
df.iloc[5:10]

In [ ]:
# Select specific rows and columns with .loc
df.loc[5:10, ["country", "year", "lifeExp"]]

**Important difference:** `.iloc[5:10]` excludes index 10 (like normal Python slicing), but `.loc[5:10]` **includes** both endpoints. This is a common source of confusion!

### Exercise 3: Selection

**3a.** Select only the `country` and `pop` columns and display the first 5 rows.

In [ ]:
# Your code here


**3b.** Use `.iloc` to select the last 3 rows of the DataFrame.

*Hint: remember negative indexing from the Python essentials notebook — `[-3:]` means "the last 3".*

In [ ]:
# Your code here


---
## 4. Filtering with Booleans

Filtering is one of the most important pandas operations. It lets you ask questions like *"which countries had a life expectancy above 80 in 2007?"*

The idea: a comparison on a column produces a **Series of booleans** (`True`/`False` for each row). When you put that Series inside `df[...]`, pandas keeps only the `True` rows.

In [ ]:
# Step 1: The comparison produces True/False for every row
df["lifeExp"] > 80

In [ ]:
# Step 2: Use that boolean Series to filter the DataFrame
long_lived = df[df["lifeExp"] > 80]
long_lived.head(10)

### Combining conditions

Use `&` (and) and `|` (or) to combine conditions. **Each condition must be in parentheses.**

Note: In the Python essentials notebook we used the words `and`/`or`. Inside pandas, we use the symbols `&`/`|` instead — they work element-wise across entire columns.

In [ ]:
# Countries with life expectancy above 80 in the year 2007
result = df[(df["lifeExp"] > 80) & (df["year"] == 2007)]
result[["country", "lifeExp"]].sort_values("lifeExp", ascending=False)

In [ ]:
# Countries in either Oceania or Europe with GDP above 30,000
rich = df[
    ((df["continent"] == "Oceania") | (df["continent"] == "Europe"))
    & (df["gdpPercap"] > 30000)
]
print(f"{len(rich)} rows match")
rich.head()

### Filtering with `.isin()`

When checking if a column matches one of several values, `.isin()` is cleaner than chaining `|` conditions.

In [ ]:
# Same as (continent == "Oceania") | (continent == "Europe"), but cleaner
df[df["continent"].isin(["Oceania", "Europe"])].head()

### Exercise 4: Filtering

**4a.** Filter the DataFrame to show only rows for the United States.

In [ ]:
# Your code here


**4b.** Find all countries in Africa where life expectancy was below 40. Which countries and years appear?

*Hint: combine two conditions with `&`, and remember the parentheses!*

In [ ]:
# Your code here


**4c.** How many rows have a GDP per capita above $10,000?

*Hint: filter first, then check the `.shape` or use `len()`.*

In [ ]:
# Your code here


---
## 5. Sorting

`.sort_values()` sorts the DataFrame by one or more columns.

In [ ]:
# 10 countries with the highest life expectancy (in any year)
df.sort_values("lifeExp", ascending=False).head(10)

In [ ]:
# Sort by multiple columns: continent first, then life expectancy within each continent
df.sort_values(["continent", "lifeExp"], ascending=[True, False]).head(10)

### Exercise 5: Sorting

**5a.** What are the 5 countries with the lowest GDP per capita in 2007?

*Hint: filter to year 2007 first, then sort by `gdpPercap` and use `.head(5)`.*

In [ ]:
# Your code here


---
## What's next?

Now that you can load, explore, filter, and sort data, we'll learn how to **visualize** it with matplotlib — turning numbers into plots that reveal patterns and tell stories. Tomorrow, we'll come back to pandas to learn about grouping and creating new columns.

## Key Points

- **Load data** with `pd.read_csv()`. Inspect it with `.head()`, `.shape`, `.info()`, `.dtypes`.
- **Summarize** with `.describe()`, `.mean()`, `.median()`, `.value_counts()`.
- **Select columns** with `df["col"]` or `df[["col1", "col2"]]`. Select rows with `.iloc[]` (by position) or `.loc[]` (by label).
- **Filter** with boolean conditions: `df[df["col"] > value]`. Combine with `&` (and) or `|` (or), and wrap each condition in parentheses.
- **Sort** with `.sort_values()`.